# EDA Meter Profile Raw

전처리 이전(raw에 가까운) 단일 계량기 상세 EDA입니다. 기본 대상은 `H1.Z16`이며, `fetch_joined_data()` 기준으로 weather join과 숫자형 변환까지만 적용합니다.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path('/home/playdata2/final_pj/energy-platform')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.meter_metadata import get_metadata
from scripts.preprocess_h1z16 import fetch_joined_data

METER_URN = 'H1.Z16'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda_raw' / METER_URN.replace('.', '_')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = get_metadata(METER_URN)
raw_df = fetch_joined_data(METER_URN).sort_values('ts').reset_index(drop=True)
raw_df['hour'] = raw_df['ts'].dt.hour
raw_df['month'] = raw_df['ts'].dt.month
raw_df['is_weekend'] = raw_df['ts'].dt.dayofweek.isin([5, 6]).astype(int)
target_col = metadata['anomaly_target']
metadata

## 1. 기초 통계 (Raw)

In [ ]:
print('rows:', len(raw_df))
print('start_ts:', raw_df['ts'].min())
print('end_ts:', raw_df['ts'].max())
display(raw_df[[c for c in [target_col, 'W', 'PF', 'Ta', 'Igm'] if c in raw_df.columns]].describe())
display(raw_df[[c for c in [target_col, 'W', 'PF', 'Ta', 'Igm'] if c in raw_df.columns]].isnull().sum())

if target_col in raw_df.columns:
    print('negative target count:', int((raw_df[target_col] < 0).sum()))
if 'W' in raw_df.columns:
    print('negative W count:', int((raw_df['W'] < 0).sum()))

## 2. 시계열 시각화 (Raw)

In [ ]:
cols = [c for c in [target_col, 'W', 'PF', 'Ta', 'Igm'] if c in raw_df.columns]
fig, axes = plt.subplots(len(cols), 1, figsize=(16, 3.5 * len(cols)), sharex=True)
if len(cols) == 1:
    axes = [axes]
for ax, col in zip(axes, cols):
    ax.plot(raw_df['ts'], raw_df[col], linewidth=0.8)
    ax.set_title(f'{METER_URN} RAW {col}')
plt.tight_layout()
plot_path = OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_raw_timeseries.png'
plt.savefig(plot_path, dpi=150)
plt.close(fig)
display(Image(filename=str(plot_path)))

## 3. 그룹 집계 (Raw)

In [ ]:
hourly = raw_df.groupby('hour')[target_col].mean().reset_index()
monthly = raw_df.groupby('month')[target_col].mean().reset_index()
weekend = raw_df.groupby('is_weekend')[target_col].mean().reset_index()
display(hourly.head())
display(monthly.head())
display(weekend)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hourly['hour'], hourly[target_col], marker='o')
axes[0].set_title('Hourly Mean (Raw)')
axes[1].bar(monthly['month'], monthly[target_col])
axes[1].set_title('Monthly Mean (Raw)')
axes[2].bar(weekend['is_weekend'].astype(str), weekend[target_col])
axes[2].set_title('Weekend vs Weekday (Raw)')
plt.tight_layout()
plot_path = OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_raw_profiles.png'
plt.savefig(plot_path, dpi=150)
plt.close(fig)
display(Image(filename=str(plot_path)))

## 4. 상관분석 / 데이터 품질 (Raw)

In [ ]:
corr_cols = [c for c in [target_col, 'PF', 'I1', 'I2', 'I3', 'P1', 'P2', 'P3', 'Ta', 'Igm'] if c in raw_df.columns and raw_df[c].notna().any()]
corr = raw_df[corr_cols].corr(numeric_only=True)
display(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar()
plt.title(f'{METER_URN} Raw Correlation Heatmap')
plt.tight_layout()
plot_path = OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_raw_corr.png'
plt.savefig(plot_path, dpi=150)
plt.close()
display(Image(filename=str(plot_path)))

quality_df = pd.DataFrame({
    'null_count': raw_df.isnull().sum(),
    'null_ratio': raw_df.isnull().mean(),
}).sort_values('null_ratio', ascending=False)
display(quality_df.head(15))
quality_df.to_csv(OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_raw_quality.csv', encoding='utf-8-sig')